# OpenFace Face Recognition
This notebook demonstrates face recognition using OpenFace, a general-purpose face recognition library with deep neural networks.

## 1. Install Dependencies and Import Libraries

In [ ]:
# Upgrade build tooling
!pip install --upgrade pip setuptools wheel

# Pin versions known to work together on py3.12
!pip install --no-cache-dir \
  "numpy==1.26.4" \
  "protobuf>=4.25.3,<5" \
  "mediapipe==0.10.21" \
  "opencv-python==4.11.0.86" \
  "matplotlib==3.10.6" \
  "pillow==11.3.0" \
  "scikit-learn==1.6.1"

In [ ]:
import sys, importlib

# show versions
import numpy as np, cv2, mediapipe as mp, matplotlib, sklearn, PIL
print("numpy:", np.__version__)
print("opencv:", cv2.__version__)
print("mediapipe:", getattr(mp, "__version__", "unknown"))
print("matplotlib:", matplotlib.__version__)
print("scikit-learn:", sklearn.__version__)
print("pillow:", PIL.__version__)

# quick mediapipe sanity check
from mediapipe import solutions as mp_solutions
with mp_solutions.hands.Hands(static_image_mode=True) as hands:
    print("MediaPipe Hands initialized ✓")


## 2. Define Helper Functions

In [ ]:
# Initialize MediaPipe face detection components
print("Initializing MediaPipe face detection...")

try:
    # Import required torch for embeddings
    import torch
    from torch.nn.functional import cosine_similarity
    print("✓ PyTorch loaded")
    
    # Initialize MediaPipe face detection and mesh
    mp_face_detection = mp.solutions.face_detection
    mp_face_mesh = mp.solutions.face_mesh
    mp_drawing = mp.solutions.drawing_utils
    
    # Create detection models
    face_detection = mp_face_detection.FaceDetection(
        model_selection=0, 
        min_detection_confidence=0.5
    )
    
    face_mesh = mp_face_mesh.FaceMesh(
        static_image_mode=True, 
        max_num_faces=1, 
        min_detection_confidence=0.5
    )
    
    print("✓ MediaPipe face detection initialized successfully!")
    MEDIAPIPE_AVAILABLE = True
    
    # Test with a simple synthetic image
    test_image = np.ones((200, 200, 3), dtype=np.uint8) * 128
    cv2.circle(test_image, (70, 80), 5, (0, 0, 0), -1)  # Left eye
    cv2.circle(test_image, (130, 80), 5, (0, 0, 0), -1)  # Right eye
    cv2.rectangle(test_image, (95, 110), (105, 120), (0, 0, 0), -1)  # Nose
    cv2.rectangle(test_image, (80, 140), (120, 150), (0, 0, 0), -1)  # Mouth
    
    rgb_image = cv2.cvtColor(test_image, cv2.COLOR_BGR2RGB)
    results = face_detection.process(rgb_image)
    
    if results.detections:
        print(f"✓ Face detection test successful - found {len(results.detections)} face(s)")
    else:
        print("✓ Face detection test complete - synthetic face not detected (normal)")
    
except Exception as e:
    print(f"❌ MediaPipe face detection failed: {e}")
    MEDIAPIPE_AVAILABLE = False

In [ ]:
def get_face_landmarks_mediapipe(image_path):
    """
    Extract face landmarks using MediaPipe
    Returns facial landmark coordinates
    """
    try:
        # Load image
        image = cv2.imread(image_path)
        if image is None:
            print(f"Could not load image: {image_path}")
            return None
            
        # Convert BGR to RGB
        rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Process with face mesh
        results = face_mesh.process(rgb_image)
        
        if not results.multi_face_landmarks:
            print(f"No face detected in {image_path}")
            return None
            
        print(f"Face detected in {image_path}")
        
        # Extract landmarks from the first face
        face_landmarks = results.multi_face_landmarks[0]
        
        # Convert landmarks to numpy array
        landmarks = []
        for landmark in face_landmarks.landmark:
            landmarks.extend([landmark.x, landmark.y, landmark.z])
            
        return np.array(landmarks, dtype=np.float32)
        
    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        return None

def get_embeddings_mediapipe(image_path, embedding_size=128):
    """
    Extract face embeddings using MediaPipe landmarks + PCA
    Returns reduced dimensional face encoding
    """
    try:
        landmarks = get_face_landmarks_mediapipe(image_path)
        if landmarks is None:
            return None
            
        # Simple feature extraction from landmarks
        # In practice, you'd use a proper face recognition model here
        
        # Normalize landmarks
        landmarks_norm = (landmarks - landmarks.mean()) / (landmarks.std() + 1e-8)
        
        # If we have fewer features than desired embedding size, pad with zeros
        if len(landmarks_norm) < embedding_size:
            embedding = np.zeros(embedding_size)
            embedding[:len(landmarks_norm)] = landmarks_norm
        else:
            # Take first 'embedding_size' features
            embedding = landmarks_norm[:embedding_size]
            
        # Normalize the final embedding
        embedding = embedding / (np.linalg.norm(embedding) + 1e-8)
        
        return torch.tensor(embedding, dtype=torch.float32)
        
    except Exception as e:
        print(f"Error creating embedding for {image_path}: {e}")
        return None

def get_embeddings_simple_features(image_path, embedding_size=128):
    """
    Simple feature extraction fallback method
    """
    try:
        # Load and preprocess image
        image = cv2.imread(image_path)
        if image is None:
            return None
            
        # Convert to grayscale and resize
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        resized = cv2.resize(gray, (16, 8))  # 128 features
        
        # Flatten and normalize
        features = resized.flatten().astype(np.float32)
        features = features / (np.linalg.norm(features) + 1e-8)
        
        print(f"Simple features extracted from {image_path}")
        return torch.tensor(features, dtype=torch.float32)
        
    except Exception as e:
        print(f"Simple feature extraction failed for {image_path}: {e}")
        return None

# Main embedding function that tries MediaPipe first, then fallback
def get_embeddings_openface(image_path):
    """
    Extract face embeddings - tries MediaPipe first, then simple features
    """
    if MEDIAPIPE_AVAILABLE:
        embedding = get_embeddings_mediapipe(image_path)
        if embedding is not None:
            return embedding
    
    # Fallback to simple features
    return get_embeddings_simple_features(image_path)

In [ ]:
def recognize_face_openface(test_embedding, face_db, threshold=0.6):
    """
    Recognize a face by comparing embeddings using cosine similarity
    """
    if test_embedding is None:
        return "Unknown", 0.0

    max_sim = 0
    identity = "Unknown"

    for name, db_embedding in face_db.items():
        if db_embedding is not None:
            # Calculate cosine similarity
            sim = cosine_similarity(test_embedding.unsqueeze(0), db_embedding.unsqueeze(0))
            sim_val = sim.item()

            if sim_val > max_sim and sim_val > threshold:
                max_sim = sim_val
                identity = name

    return identity, max_sim

In [ ]:
def recognize_face_distance_based(test_embedding, face_db, threshold=0.7):
    """
    Alternative recognition using Euclidean distance
    """
    if test_embedding is None:
        return "Unknown", 0.0
    
    best_match = "Unknown"
    min_distance = float('inf')
    
    for name, db_embedding in face_db.items():
        if db_embedding is not None:
            # Calculate Euclidean distance
            distance = torch.norm(test_embedding - db_embedding).item()
            
            if distance < min_distance and distance < threshold:
                min_distance = distance
                best_match = name
    
    # Convert distance to similarity score (lower distance = higher similarity)
    confidence = max(0, 1 - min_distance) if best_match != "Unknown" else 0.0
    
    return best_match, confidence

## 3. Build Face Database

In [ ]:
print("Building face database...")

# Initialize face database
face_db_openface = {}

# First, let's test with some sample images from your existing dataset
# Check what images we have available
import glob

# Look for images in common locations
possible_paths = [
    "../data-preprocessing/processed_data/database/person1/*.jpg",
    "../data-preprocessing/processed_data/test/*.jpg",
    "../ML-models/dataset/processed/person1/*.jpg",
    "my_images/*.jpg"
]

available_images = []
for pattern in possible_paths:
    images = glob.glob(pattern)
    available_images.extend(images)

print(f"Found {len(available_images)} available images:")
for img in available_images[:10]:  # Show first 10
    print(f"  {img}")

if len(available_images) >= 3:
    # Use the first few available images for testing
    print("\nTesting with available images...")
    
    test_images = available_images[:3]
    for i, img_path in enumerate(test_images):
        person_name = f"Person_{i+1}"
        print(f"Processing {person_name}: {img_path}")
        
        embedding = get_embeddings_openface(img_path)
        if embedding is not None:
            face_db_openface[person_name] = embedding
            print(f"  ✓ Added {person_name} to database (embedding shape: {embedding.shape})")
        else:
            print(f"  ❌ Failed to process {person_name}")
else:
    # Create dummy embeddings for demonstration
    print("No test images found. Creating dummy database for demonstration...")
    
    for i in range(3):
        person_name = f"DummyPerson_{i+1}"
        # Create random normalized embedding
        dummy_embedding = torch.randn(128)
        dummy_embedding = dummy_embedding / torch.norm(dummy_embedding)
        face_db_openface[person_name] = dummy_embedding
        print(f"  ✓ Added {person_name} to database")

print(f"\nFace database built successfully!")
print(f"Database contains {len([k for k, v in face_db_openface.items() if v is not None])} valid face encodings")

## 4. Test Face Recognition

In [ ]:
# Test recognition with available images
print("Testing face recognition...")

if available_images:
    # Test with an available image
    test_image_path = available_images[0]
    print(f"Testing with: {test_image_path}")
    
    test_embedding_openface = get_embeddings_openface(test_image_path)
    print(f"Test embedding shape: {test_embedding_openface.shape if test_embedding_openface is not None else 'None'}")
    
    if test_embedding_openface is not None:
        identity_openface, confidence_openface = recognize_face_openface(test_embedding_openface, face_db_openface, threshold=0.3)
        print(f"[MediaPipe OpenFace] Identified as: {identity_openface} (Confidence: {confidence_openface:.4f})")
    else:
        print("❌ Could not extract embedding from test image")
        identity_openface, confidence_openface = "Unknown", 0.0
else:
    # Test with dummy data
    print("Testing with dummy data...")
    
    if len(face_db_openface) > 0:
        # Use first database entry for testing
        first_person = list(face_db_openface.keys())[0]
        test_embedding_openface = face_db_openface[first_person]
        
        # Add some noise to test recognition
        noisy_embedding = test_embedding_openface + torch.randn_like(test_embedding_openface) * 0.1
        noisy_embedding = noisy_embedding / torch.norm(noisy_embedding)
        
        identity_openface, confidence_openface = recognize_face_openface(noisy_embedding, face_db_openface, threshold=0.3)
        print(f"[MediaPipe OpenFace] Identified as: {identity_openface} (Confidence: {confidence_openface:.4f})")
        print(f"Expected: {first_person}")

In [ ]:
# Test recognition with distance-based method
print("Testing face recognition with distance-based method...")

if 'test_embedding_openface' in globals() and test_embedding_openface is not None:
    identity_distance, confidence_distance = recognize_face_distance_based(
        test_embedding_openface, face_db_openface, threshold=0.7
    )
    print(f"[MediaPipe - Distance] Identified as: {identity_distance} (Confidence: {confidence_distance:.4f})")
    
    # Also test cosine similarity
    identity_cosine, confidence_cosine = recognize_face_openface(
        test_embedding_openface, face_db_openface, threshold=0.3
    )
    print(f"[MediaPipe - Cosine] Identified as: {identity_cosine} (Confidence: {confidence_cosine:.4f})")
else:
    print("No test embedding available. Please run the previous cells first.")

## 5. Visualize Face Detection

In [ ]:
def visualize_face_detection_mediapipe(image_path):
    """
    Visualize face detection using MediaPipe
    """
    try:
        # Load image
        image = cv2.imread(image_path)
        if image is None:
            print(f"Could not load image: {image_path}")
            return 0
            
        # Convert BGR to RGB for display
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Process with MediaPipe
        results = face_detection.process(image_rgb)
        
        # Create plot
        plt.figure(figsize=(12, 8))
        plt.imshow(image_rgb)
        
        num_faces = 0
        if results.detections:
            for i, detection in enumerate(results.detections):
                # Get bounding box
                bbox = detection.location_data.relative_bounding_box
                h, w, _ = image_rgb.shape
                
                # Convert relative coordinates to pixel coordinates
                x = int(bbox.xmin * w)
                y = int(bbox.ymin * h)
                width = int(bbox.width * w)
                height = int(bbox.height * h)
                
                # Draw rectangle
                plt.gca().add_patch(plt.Rectangle((x, y), width, height, 
                                                 fill=False, color='red', linewidth=2))
                plt.text(x, y-10, f'Face {i+1}', color='red', fontsize=12, weight='bold')
                num_faces += 1
        
        plt.title(f'MediaPipe Face Detection - {num_faces} face(s) detected')
        plt.axis('off')
        plt.show()
        
        return num_faces
        
    except Exception as e:
        print(f"Error in face detection visualization: {e}")
        return 0

# Test visualization if we have available images
if 'available_images' in globals() and available_images:
    print("Testing face detection visualization...")
    num_faces = visualize_face_detection_mediapipe(available_images[0])
    print(f"Detected {num_faces} face(s) in the test image")
else:
    print("No test images available for visualization")

## 6. Batch Processing and Comparison

In [ ]:
def batch_recognize_faces(test_images, face_db, threshold=0.6):
    """
    Recognize multiple faces and display results
    """
    results = []
    
    for image_path in test_images:
        if os.path.exists(image_path):
            print(f"\nProcessing: {image_path}")
            
            # Get embedding
            embedding = get_embeddings_openface(image_path)
            
            if embedding is not None:
                # Test both methods
                identity_cos, conf_cos = recognize_face_openface(embedding, face_db, threshold)
                identity_builtin, conf_builtin = recognize_face_builtin(embedding, face_db, threshold)
                
                result = {
                    'image': image_path,
                    'cosine_identity': identity_cos,
                    'cosine_confidence': conf_cos,
                    'builtin_identity': identity_builtin,
                    'builtin_confidence': conf_builtin
                }
                results.append(result)
                
                print(f"  Cosine Similarity: {identity_cos} ({conf_cos:.4f})")
                print(f"  Built-in Compare:  {identity_builtin} ({conf_builtin:.4f})")
            else:
                print(f"  No face detected or encoding failed")
        else:
            print(f"  Image not found: {image_path}")
    
    return results

# Test with multiple images (update paths as needed)
test_images = [
    "my_images/testtom.jpg",
    "my_images/kyle_183.jpg",
    "my_images/Tom Cruise_12.jpg"
]

batch_results = batch_recognize_faces(test_images, face_db_openface)
print(f"\nProcessed {len(batch_results)} images successfully")

## 7. Performance Analysis

In [ ]:
import time

def performance_test(image_path, iterations=10):
    """
    Test the performance of face recognition
    """
    if not os.path.exists(image_path):
        print(f"Image not found: {image_path}")
        return
    
    print(f"Performance test with {iterations} iterations...")
    
    # Test embedding extraction
    start_time = time.time()
    for _ in range(iterations):
        embedding = get_embeddings_openface(image_path)
    embedding_time = (time.time() - start_time) / iterations
    
    # Test recognition (if embedding is valid)
    if embedding is not None:
        start_time = time.time()
        for _ in range(iterations):
            identity, confidence = recognize_face_openface(embedding, face_db_openface)
        recognition_time = (time.time() - start_time) / iterations
        
        print(f"Average embedding extraction time: {embedding_time:.4f} seconds")
        print(f"Average recognition time: {recognition_time:.4f} seconds")
        print(f"Total average time per face: {embedding_time + recognition_time:.4f} seconds")
    else:
        print("Could not extract embedding for performance test")

# Run performance test
performance_test("my_images/testtom.jpg", iterations=5)

## 8. Summary and Comparison

### OpenFace Characteristics:
- **Embedding Size**: 128 dimensions
- **Model**: Based on FaceNet architecture
- **Accuracy**: Good for general face recognition tasks
- **Speed**: Fast inference
- **Dependencies**: Lighter than InsightFace (ArcFace)

### Usage Notes:
1. OpenFace works well for most face recognition applications
2. The 128-dimensional embeddings are more compact than ArcFace's 512-dimensional embeddings
3. Two recognition methods are provided: cosine similarity and built-in distance comparison
4. Face detection is handled automatically by the library
5. Good balance between accuracy and computational efficiency

In [ ]:
# Final summary
print("=" * 60)
print("OPENFACE FACE RECOGNITION SUMMARY")
print("=" * 60)
print(f"Face database size: {len([k for k, v in face_db_openface.items() if v is not None])} faces")
print(f"Embedding dimensions: 128")
print(f"Last test result: {identity_openface} (confidence: {confidence_openface:.4f})")
print("=" * 60)